# FIGS — Report Database Overview

Explore the combined storm-report database (IEM LSRs + SPC daily CSV + SVRGIS,
incl. post-survey EF and tornado **tracks**) over the analysis period: counts,
temporal cycles, spatial distribution, intensity distributions vs the FIGS bins,
and how many forecast hours qualify at each `min_reports` threshold.

Note: the first pass fetches one CSV per day (cached afterwards). Start with a
season; widen `START`/`END` to the full 2020-2023 once cached.

In [ ]:
# Run with the `met` conda env kernel.
%load_ext autoreload
%autoreload 2          # pick up edits to figs/ without restarting the kernel
import warnings; warnings.filterwarnings('ignore')
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '..')   # so `import figs` works from notebooks/
from figs import config as C

In [ ]:
from datetime import datetime, timezone, timedelta
from figs.data import reports, grid
from figs.config import INTENSITY_BINS, SEVERE_THRESHOLDS, SIGNIFICANT_THRESHOLDS, REPORTS_CACHE
import glob, os

# DEFAULT: read only what's ALREADY cached under Data/reports/ (instant, no network).
# Your dataset builds cache every severe day there, so this shows the ingested DB as-is.
USE_CACHE_ONLY = True

if USE_CACHE_ONLY:
    files = sorted(glob.glob(str(REPORTS_CACHE / 'reports_*.csv')))
    print('cached day-files:', len(files))
    frames = [pd.read_csv(f, parse_dates=['time']) for f in files if os.path.getsize(f) > 0]
else:
    # fetch an explicit window (fetches+caches any missing days)
    START = datetime(2020, 12, 2, tzinfo=timezone.utc)
    END   = datetime(2023, 12, 31, tzinfo=timezone.utc)
    days = pd.date_range(START, END, freq='D', tz='UTC')
    frames = []
    for i, d in enumerate(days):
        try: frames.append(reports.reports_for_day(d.to_pydatetime()))
        except Exception as e: print('skip', d.date(), e)
        if i % 30 == 0: print(f'{i}/{len(days)} days')

rep = pd.concat(frames, ignore_index=True)
rep['time'] = pd.to_datetime(rep['time'], utc=True)
rep = rep.dropna(subset=['hazard'])
START, END = rep.time.min(), rep.time.max()
print('total reports', len(rep), '| period', START.date(), '->', END.date())

## Counts by hazard (severe-filtered, FIGS thresholds)

In [ ]:
# apply FIGS severe thresholds: tor any, wind>=50kt, hail>=1in
sev = rep[((rep.hazard=='tor')) | ((rep.hazard=='wind')&(rep.magnitude>=SEVERE_THRESHOLDS['wind_kt']))
          | ((rep.hazard=='hail')&(rep.magnitude>=SEVERE_THRESHOLDS['hail_in']))].copy()
print(sev.hazard.value_counts())
print('\nsignificant:')
print('  tor EF>=2:', int(((sev.hazard=='tor')&(sev.ef>=2)).sum()))
print('  wind>=65kt:', int(((sev.hazard=='wind')&(sev.magnitude>=65)).sum()))
print('  hail>=2in :', int(((sev.hazard=='hail')&(sev.magnitude>=2)).sum()))

## Temporal cycles — by month and hour of day (UTC)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,4))
for h in ['tor','wind','hail']:
    s = sev[sev.hazard==h]
    s.groupby(s.time.dt.month).size().reindex(range(1,13), fill_value=0).plot(ax=ax[0], marker='o', label=h)
    s.groupby(s.time.dt.hour).size().reindex(range(24), fill_value=0).plot(ax=ax[1], marker='o', label=h)
ax[0].set_xlabel('month'); ax[0].set_title('reports by month'); ax[0].legend()
ax[1].set_xlabel('hour UTC'); ax[1].set_title('diurnal cycle'); ax[1].legend()
plt.tight_layout(); plt.show()

## Spatial distribution

In [ ]:
import cartopy.crs as ccrs, cartopy.feature as cfeature
fig = plt.figure(figsize=(12,7)); ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5))
ax.add_feature(cfeature.STATES, lw=0.3); ax.set_extent([-120,-74,23,50])
colors = {'tor':'red','wind':'blue','hail':'green'}
for h,c in colors.items():
    s = sev[sev.hazard==h]
    ax.scatter(s.lon, s.lat, s=4, c=c, alpha=0.3, transform=ccrs.PlateCarree(), label=h)
ax.set_title(f'severe reports {START:%Y-%m-%d} to {END:%Y-%m-%d}'); plt.legend(); plt.show()

## Intensity distributions vs the FIGS conditional-intensity bins

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))
# tornado EF (exclude EFU/-1)
tor = sev[(sev.hazard=='tor')&(sev.ef>=0)]
tor.ef.value_counts().sort_index().plot.bar(ax=ax[0]); ax[0].set_title('tornado EF'); ax[0].set_xlabel('EF')
# wind kt with bin edges
sev[sev.hazard=='wind'].magnitude.plot.hist(bins=40, ax=ax[1]); ax[1].set_title('wind (kt)')
for e in INTENSITY_BINS['wind']['edges']: ax[1].axvline(e, color='k', ls='--', lw=0.6)
# hail inches with bin edges
sev[sev.hazard=='hail'].magnitude.plot.hist(bins=40, ax=ax[2]); ax[2].set_title('hail (in)')
for e in INTENSITY_BINS['hail']['edges']: ax[2].axvline(e, color='k', ls='--', lw=0.6)
plt.tight_layout(); plt.show()

## SVRGIS tornado tracks (a chosen day)

In [ ]:
DAY = datetime(2023, 3, 31, 22, tzinfo=timezone.utc)  # set to a tornado day in your window
tr = reports.svrgis_tracks_in_window(DAY, 180)  # +/- 3 h
print(len(tr), 'tracks')
fig = plt.figure(figsize=(11,7)); ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5))
ax.add_feature(cfeature.STATES, lw=0.3); ax.set_extent([-105,-80,30,45])
for _,t in tr.iterrows():
    ax.plot([t.slon,t.elon],[t.slat,t.elat], '-', lw=1+ (t.ef if t.ef>0 else 0),
            color='purple', transform=ccrs.PlateCarree())
ax.set_title(f'SVRGIS tornado tracks near {DAY:%Y-%m-%d %HZ}'); plt.show()

## Qualifying forecast hours vs `min_reports` (drives sample/band counts)

Computed from the loaded (cached) reports — no network. This is the per-band
sample count for a build; multiply by the number of bands for the total.

In [ ]:
# count reports per top-of-hour across all hazards (matches severe_valid_hours)
hourly = rep.groupby(rep.time.dt.floor('h')).size()
for mr in [1,3,5,10,15,20]:
    n = int((hourly >= mr).sum())
    print(f'min_reports={mr:2d} -> {n} valid hours -> {n} samples/band')